

# **Cat vs Dog Image Classification using Transfer Learning (InceptionV3)**


**1️⃣ Import Libraries**

In [ ]:
# =========================
# 1️⃣ Import Libraries
# =========================
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing import image

**2️⃣ Load Pre-trained Model (InceptionV3)**

We use InceptionV3 as a feature extractor. We freeze the convolutional base to prevent training on ImageNet weights

In [ ]:
# Load InceptionV3 without top classifier layers
pre_trained_model = tf.keras.applications.InceptionV3(
    input_shape=(150,150,3),
    include_top=False,
    weights='imagenet'
)

# Freeze the convolutional layers
for layer in pre_trained_model.layers:
    layer.trainable = False

# Check the model summary
pre_trained_model.summary()

In [ ]:
# Choose last layer for feature extraction
last_layer = pre_trained_model.get_layer('mixed7')
print('Last layer output shape:', last_layer.output.shape)
last_output = last_layer.output

**3️⃣ Build Custom Classifier**

We add Flatten, Dense, Dropout, and Output layer for binary classification (Cat vs Dog).

In [ ]:
x = tf.keras.layers.Flatten()(last_output)
x = tf.keras.layers.Dense(1024, activation='relu')(x)
x = tf.keras.layers.Dropout(0.2)(x)
x = tf.keras.layers.Dense(1, activation='sigmoid')(x)

# Combine base model and custom classifier
model = tf.keras.Model(pre_trained_model.input, x)
model.summary()

**4️⃣ Prepare Dataset**

In [ ]:
BASE_DIR = '/content/data'
train_dir = os.path.join(BASE_DIR, 'train')
validation_dir = os.path.join(BASE_DIR, 'validation')

# Prepare training dataset
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(150, 150),
    batch_size=20,
    label_mode='binary'
)

# Prepare validation dataset
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    validation_dir,
    image_size=(150, 150),
    batch_size=20,
    label_mode='binary'
)

**5️⃣ Preprocessing**

In [ ]:
# Preprocess input for InceptionV3
def preprocess(image, label):
    image = tf.keras.applications.inception_v3.preprocess_input(image)
    return image, label

train_dataset_scaled = train_dataset.map(preprocess)
validation_dataset_scaled = validation_dataset.map(preprocess)

# Optimize data pipeline
SHUFFLE_BUFFER_SIZE = 1000
PREFETCH_BUFFER_SIZE = tf.data.AUTOTUNE

train_dataset_final = (
    train_dataset_scaled
    .cache()
    .shuffle(SHUFFLE_BUFFER_SIZE)
    .prefetch(PREFETCH_BUFFER_SIZE)
)

validation_dataset_final = (
    validation_dataset_scaled
    .cache()
    .prefetch(PREFETCH_BUFFER_SIZE)
)

**6️⃣ Data Augmentation**

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.4),
    tf.keras.layers.RandomTranslation(0.2,0.2),
    tf.keras.layers.RandomContrast(0.4),
    tf.keras.layers.RandomZoom(0.2),
])

# Apply augmentation to model input
inputs = tf.keras.Input(shape=(150, 150, 3))
x = data_augmentation(inputs)
x = model(x)
model_with_aug = tf.keras.Model(inputs, x)

**7️⃣ Compile Model**

In [ ]:
model_with_aug.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

**8️⃣ Train Model**

In [ ]:
EPOCHS = 20

history = model_with_aug.fit(
    train_dataset_final,
    validation_data=validation_dataset_final,
    epochs=EPOCHS,
    verbose=2
)

**9️⃣ Plot Training Results**

In [ ]:
def plot_loss_acc(history):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']

    epochs = range(len(acc))

    fig, ax = plt.subplots(1,2, figsize=(12,6))
    ax[0].plot(epochs, acc, 'bo', label='Training accuracy')
    ax[0].plot(epochs, val_acc, 'b', label='Validation accuracy')
    ax[0].set_title('Training and validation accuracy')
    ax[0].legend()

    ax[1].plot(epochs, loss, 'bo', label='Training loss')
    ax[1].plot(epochs, val_loss, 'b', label='Validation loss')
    ax[1].set_title('Training and validation loss')
    ax[1].legend()

    plt.show()

plot_loss_acc(history)

**🔟 Test on a Single Image**

In [ ]:
class_names = ['Cat', 'Dog']

img_path = '/content/FELV-cat.jpg'
img = image.load_img(img_path, target_size=(150,150))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension

# Preprocess input
img_array = tf.keras.applications.inception_v3.preprocess_input(img_array)

pred = model_with_aug.predict(img_array)
print("Raw prediction:", pred)

if pred[0][0] > 0.5:
    print("Predicted label: Dog")
else:
    print("Predicted label: Cat")